# Building RAG Chatbots with LangChain

In this example, we'll work on building an AI chatbot from start-to-finish. We will be using LangChain, OpenAI, and Pinecone vector DB, to build a chatbot capable of learning from the external world using **R**etrieval **A**ugmented **G**eneration (RAG).

We will be using a dataset sourced from the Llama 2 ArXiv paper and other related papers to help our chatbot answer questions about the latest and greatest in the world of GenAI.

By the end of the example we'll have a functioning chatbot and RAG pipeline that ***can hold a conversation and provide informative responses based on a knowledge base.***

### Before you begin

You'll need to install [Ollama](https://ollama.com/), a program that runs open models locally, and [ChromaDB](https://www.trychroma.com/products/chromadb/), a vector database that can be run locally. Alternative databases are Redis and Postgres, which both are capable of computing cosine similarity.

### Prerequisites

Before we start building our chatbot, we need to install some Python libraries. Here's a brief overview of what each library does:

- **langchain**: This is a library for GenAI. We'll use it to chain together different language models and components for our chatbot. It has components to connect to Ollama and to Chroma.
- **datasets**: This library provides a vast array of datasets for machine learning. We'll use it to load our knowledge base for the chatbot.

You can install these libraries using pip like so:

In [1]:
!pip install -qU chromadb langchain-community langchain-ollama langchain-chroma langchain-huggingface

In [2]:
import os

In [3]:
import textwrap

In [4]:
import time

In [5]:
from tqdm.auto import tqdm  # for progress bar

In [6]:
from datasets import load_dataset

In [7]:
from langchain_core.chat_history import (
    BaseChatMessageHistory, # Interface/typing for pluggable chat history backends.
    InMemoryChatMessageHistory # for in-memory chat history for the session.
)
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # for build chat-style prompts; placeholder to inject prior messages/history.
from langchain_core.runnables.history import RunnableWithMessageHistory # to wrap a chain so it automatically reads/writes chat history per session id.
from langchain_text_splitters import RecursiveCharacterTextSplitter # doc splits into overlapping chunks to improve retrieval quality.
# from langchain_community.document_loaders import PyPDFLoader # for loading and parsing pdf files in to lanchain doc objects

from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

### Building a Chatbot (no RAG)

We will be relying heavily on the LangChain library to bring together the different components needed for our chatbot. To begin, we'll create a simple chatbot without any retrieval augmentation.

In [8]:
from langchain_ollama import ChatOllama

In [9]:
chat = ChatOllama(
    model="phi3:mini"
)

Chats with AI chat models are typically structured (in plain text) like this:

```
System: You are a helpful assistant.

User: Hi AI, how are you today?

Assistant: I'm great thank you. How can I help you?

User: I'd like to understand string theory.

Assistant:
```

The final `"Assistant:"` without a response is what would prompt the model to continue the conversation. In the official OpenAI `ChatCompletion` endpoint these would be passed to the model in a format like:

```python
[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi AI, how are you today?"},
    {"role": "assistant", "content": "I'm great thank you. How can I help you?"}
    {"role": "user", "content": "I'd like to understand string theory."}
]
```

In LangChain there is a slightly different format. We use three _message_ objects like so:

In [10]:
messages = [
    SystemMessage(content="You are a helpful assistant. Please make your responses brief."),
    HumanMessage(content="Hi AI, how are you today?"),
    AIMessage(content="I'm great thank you. How can I help you?"),
    HumanMessage(content="I'd like to understand string theory.")
]

The format is very similar, we just swapped the role of `"user"` for `HumanMessage`, and the role of `"assistant"` for `AIMessage`.

We generate the next response from the AI by passing these messages to the `ChatOpenAI` object.

In [11]:
res = chat.invoke(messages)
res

AIMessage(content='Of course! String theory is a theoretical framework where the point-like particles of particle physics are replaced by one-dimensional objects known as "strings." It describes how these strings propagate through space and interact with each other, which could unify all fundamental forces including gravity into one single force.', additional_kwargs={}, response_metadata={'model': 'phi3:mini', 'created_at': '2026-11-04T22:28:02.324597468Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10994091125, 'load_duration': 10351422072, 'prompt_eval_count': 68, 'prompt_eval_duration': 215358143, 'eval_count': 60, 'eval_duration': 352306147, 'logprobs': None, 'model_name': 'phi3:mini', 'model_provider': 'ollama'}, id='lc_run--01a1a8ac-fb9c-73f1-870e-ba78aff05c8e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 68, 'output_tokens': 60, 'total_tokens': 128})

In response we get another AI message object. We can print it more clearly like so:

In [12]:
print(textwrap.fill(res.content, 100))

Of course! String theory is a theoretical framework where the point-like particles of particle
physics are replaced by one-dimensional objects known as "strings." It describes how these strings
propagate through space and interact with each other, which could unify all fundamental forces
including gravity into one single force.


Because `res` is just another `AIMessage` object, we can append it to `messages`, add another `HumanMessage`, and generate the next response in the conversation.

In [13]:
# add latest AI response to messages
messages.append(res)

# now create a new user prompt
prompt = HumanMessage(
    content="Why do physicists believe it can produce a 'unified theory'?"
)

# add to messages
messages.append(prompt)

# send to AI model
res = chat.invoke(messages)

In [14]:
print(textwrap.fill(res.content, 100))

String theorists propose that by considering particles as tiny vibrating strings rather than points,
we might be able to reconcile quantum mechanics with general relativity - two fundamental theories
in physics which are currently not fully compatible. This unification is known as a 'Theory of
Everything' and string theory has been one leading candidate due to its potential ability to
incorporate gravity into the framework of particle physics.


### Dealing with Hallucinations

We have our chatbot, but as mentioned — the knowledge of LLMs can be limited. The reason for this is that LLMs learn all they know during training. An LLM essentially compresses the "world" as seen in the training data into the internal parameters of the model. We call this knowledge the _parametric knowledge_ of the model.

By default, LLMs have no access to the external world.

The result of this is very clear when we ask LLMs about more recent information, like about the Llama 2 LLM.

In [15]:
# add latest AI response to messages
messages.append(res)

# now create a new user prompt
prompt = HumanMessage(
    content="What is so special about Gemma 4?"
)

# add to messages
messages.append(prompt)

# send to model
res = chat.invoke(messages)

In [16]:
print(textwrap.fill(res.content, 100))

I believe you might be referring to an artificial intelligence program called "Gemma." If this AI
exists, it's designed for creative writing and generating human-like text with advanced natural
language processing capabilities like understanding context and sentiment. The technology behind
such a system combines machine learning algorithms that analyze vast amounts of data (text in
particular) to learn patterns and apply them when creating new content.


The Phi 3 Mini model has a knowledge cutoff date of October 2023. This means it doesn't have information on developments that occurred after that time, including the release of Google's Gemma 4 in April 2026. Therefore, Phi 3 Mini would not be aware of Gemma 4 or any related advancements.

Our chatbot can no longer help us, it doesn't contain the information we need to answer the question. It was very clear from this answer that the LLM doesn't know the informaiton, but sometimes an LLM may respond like it _does_ know the answer — and this can be very hard to detect.

In [17]:
# add latest AI response to messages
messages.append(res)

# now create a new user prompt
prompt = HumanMessage(
    content="Can you tell me about Matryoshka Formers?"
)
# add to messages
messages.append(prompt)

# send to AI
res = chat.invoke(messages)

In [18]:
print(textwrap.fill(res.content, 100))

I apologize for the confusion, but there isn't any widely recognized concept or technology called
"Matryoshka Formers" that I am aware of as of my last update in early 2023. If this term relates to
a specific niche topic, such as Russian dolls (Matryoshka) applied metaphorically within the field
you're asking about – perhaps nested architectures or multi-layered models in AI? Let me know if
that seems like what you were referring to!


There is another way of feeding knowledge into LLMs. It is called _source knowledge_ and it refers to any information fed into the LLM via the prompt. We can try that with the LLMChain question. We can take a description of this object from the LangChain documentation.

In [19]:
matformer_information = [
    "MatFormer (short for Matryoshka Transformer) is a natively elastic model architecture that incorporates nested parameter sub-structures directly inside standard transformer blocks.",
    "By optimizing these nested structures—primarily within the Feed-Forward Network layers—a single base model can dynamically yield hundreds of accurate smaller sub-models without requiring separate training runs.",
    "This elastic design allows deployment systems to slice and adjust parameter sizes on the fly to meet real-time compute, memory, and latency constraints."
]

source_knowledge = "\n".join(matformer_information)

We can feed this additional knowledge into our prompt with some instructions telling the LLM how we'd like it to use this information alongside our original query.

In [20]:
query = "Can you tell me about Matryoshka Formers?"

augmented_prompt = f"""Using the contexts below, answer the query.

Contexts:
{source_knowledge}

Query: {query}"""

Now we feed this into our chatbot as we were before.

In [21]:
# create a new user prompt
prompt = HumanMessage(
    content=augmented_prompt
)
# add to messages
messages.append(prompt)

# send to AI
res = chat.invoke(messages)

In [22]:
print(textwrap.fill(res.content, 100))

The MatFormer is an advanced neural network architecture inspired by Russian nesting dolls
(Matryoshka). It introduces a way for the model to create smaller nested models dynamically within
its structure, which can be tailored without separate training. The flexibility of this design
facilitates deployments where computational resources need to adapt in real-time based on current
constraints and demands.


The quality of this answer is phenomenal. This is made possible thanks to the idea of augmented our query with external knowledge (source knowledge). There's just one problem — how do we get this information in the first place?

We will use a vector database. But first, we'll need a dataset.

### Importing the Data

In this task, we will be importing our data. We will be using the Hugging Face Datasets library to load our data. Specifically, we will be using the `yufan/arxiv-metadata-2020-2026` dataset. This dataset contains a collection of ArXiv papers which will serve as the external knowledge base for our chatbot.

In [24]:
dataset = load_dataset(
    "yufan/arxiv-metadata-2020-2026",
    "Computer_Science",
    split='2023',
)

In [25]:
dataset

Dataset({
    features: ['arxiv_id', 'corpusId', 'title', 'authors', 'year', 'publicationDate', 'citationCount', 'influentialCitationCount', 'fieldsOfStudy', 'abstract', 'fetched_at', 'citation_refreshed_at', 'citation_source_release', 'company'],
    num_rows: 93422
})

#### Dataset Overview

It is a collection of academic papers from ArXiv, a repository of electronic preprints approved for publication after moderation. Each entry in the dataset represents a "chunk" of text from these papers.

Because most **L**arge **L**anguage **M**odels (LLMs) only contain knowledge of the world as it was during training, they cannot answer our questions about Llama 2 — at least not without this data.

### Task 4: Building the Knowledge Base

We now have a dataset that can serve as our chatbot knowledge base. Our next task is to transform that dataset into the knowledge base that our chatbot can use. To do this we must use an embedding model and vector database.

We begin by initializing our connection to Pinecone, this requires a [free API key](https://app.pinecone.io).

Then we initialize the index. We will be using OpenAI's `text-embedding-ada-002` model for creating the embeddings, so we set the `dimension` to `1536`.

In [23]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [24]:
embed_model = "mixedbread-ai/mxbai-embed-large-v1"
embeddings = HuggingFaceEmbeddings(
    model_name=embed_model,
    model_kwargs={'device': 'cuda:1'},
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [25]:
index_name = 'arxiv-rag' # data structure that organizes high-dimensional vectors to enable very fast retrievval/ similarity searches

In [26]:
# initialize the vector store object
vector_store = Chroma(
    collection_name=index_name,
    embedding_function=embeddings,
    host="localhost",
)

In [75]:
vector_store._collection.count()

0

Using this model we can create embeddings like so:

In [76]:
texts = [
    'this is the first chunk of text',
    'then another second chunk of text is here'
]

res = embeddings.embed_documents(texts)
len(res), len(res[0])

(2, 1024)

In [77]:
dimensions = len(res[0])

From this we get two (aligning to our two chunks of text) 1536-dimensional embeddings.

We're now ready to embed and index all our our data! We do this by looping through our dataset and embedding and inserting everything in batches.

In [78]:
dataset[0]

{'arxiv_id': '2301.00001',
 'corpusId': 255373121,
 'title': 'NFTrig',
 'authors': 'Jordan J. Thompson; Ryan Benac; Kidus Olana; Talha Hassan; Andrew Sward; Tauheed Khan Mohd',
 'year': 2022,
 'publicationDate': datetime.datetime(2022, 12, 21, 0, 0),
 'citationCount': 0,
 'influentialCitationCount': 0,
 'fieldsOfStudy': 'Computer Science',
 'abstract': 'NFTrig is a web-based application created for use as an educational tool to teach trigonometry and block chain technology. Creation of the application includes front and back end development as well as integration with other outside sources including MetaMask and OpenSea. The primary development languages include HTML, CSS (Bootstrap 5), and JavaScript as well as Solidity for smart contract creation. The application itself is hosted on Moralis utilizing their Web3 API. This technical report describes how the application was created, what the application requires, and smart contract design with security considerations in mind. The NFTrig

In [79]:
data = dataset.to_pandas()  # this makes it easier to iterate over the dataset
data = data[~(data['abstract'].isnull() | data['title'].isnull())]

batch_size = 100

for i in tqdm(range(0, len(data), batch_size)):
    i_end = min(len(data), i + batch_size)
    # get batch of data
    batch = data.iloc[i:i_end]
    
    # generate unique ids for each chunk
    ids = [f"{x['corpusId']}-{x['arxiv_id']}" for i, x in batch.iterrows()]
    
    # get text to embed
    texts = [x['title'] + '\n\n' + x['abstract'] for _, x in batch.iterrows()]
    
    # embed text
    embeds = embeddings.embed_documents(texts)

    documents = [Document(text) for text in texts]
    
    # add to VDB
    vector_store.add_documents(documents=documents, ids=ids)

  0%|          | 0/935 [00:00<?, ?it/s]

We can check that the vector index has been populated:

In [27]:
vector_store._collection.count()

93421

#### Retrieval Augmented Generation

We've built a fully-fledged knowledge base. Now it's time to connect that knowledge base to our chatbot. To do that we'll be diving back into LangChain and reusing our template prompt from earlier.

To use LangChain with your data, we need to tell LangChain where your data is stored and how to access it. This is done through a "vector store" abstraction.
We pass in our vector index to initialize the object.

Using this `vector_store` we can already query the index and see if we have any relevant information given our question about MatFormers.

In [28]:
query = "What is a MatFormer?"

vector_store.similarity_search(query, k=3)

[Document(id='263834773-2310.07707', metadata={}, page_content='MatFormer: Nested Transformer for Elastic Inference\n\nFoundation models are applied in a broad spectrum of settings with different inference constraints, from massive multi-accelerator clusters to resource-constrained standalone mobile devices. However, the substantial costs associated with training these models often limit the number of unique model sizes that can be offered. Consequently, practitioners are compelled to select a model that may not be optimally aligned with their specific latency and cost requirements. We present MatFormer, a novel Transformer architecture designed to provide elastic inference across diverse deployment constraints. MatFormer achieves this by incorporating a nested Feed Forward Network (FFN) block structure within a standard Transformer model. During training, we optimize the parameters of multiple nested FFN blocks with varying sizes, enabling the extraction of hundreds of accurate smalle

We return a lot of text here and it's not that clear what we need or what is relevant. Fortunately, our LLM will be able to parse this information much faster than us. All we need is to connect the output from our `vector_store` to our `chat` chatbot. To do that we can use the same logic as we used earlier.

In [29]:
def augment_prompt(query: str):
    # get top 3 results from knowledge base
    results = vector_store.similarity_search(query, k=3)
    # get the text from the results
    source_knowledge = "\n".join([x.page_content for x in results])
    # feed into an augmented prompt
    augmented_prompt = f"""Using the contexts below, answer the query.

    Contexts:
    {source_knowledge}

    Query: {query}"""
    return augmented_prompt

Using this we produce an augmented prompt:

In [30]:
augmented_prompt = augment_prompt(query)

print(augmented_prompt)

Using the contexts below, answer the query.

    Contexts:
    MatFormer: Nested Transformer for Elastic Inference

Foundation models are applied in a broad spectrum of settings with different inference constraints, from massive multi-accelerator clusters to resource-constrained standalone mobile devices. However, the substantial costs associated with training these models often limit the number of unique model sizes that can be offered. Consequently, practitioners are compelled to select a model that may not be optimally aligned with their specific latency and cost requirements. We present MatFormer, a novel Transformer architecture designed to provide elastic inference across diverse deployment constraints. MatFormer achieves this by incorporating a nested Feed Forward Network (FFN) block structure within a standard Transformer model. During training, we optimize the parameters of multiple nested FFN blocks with varying sizes, enabling the extraction of hundreds of accurate smaller m

There is still a lot of text here, so let's pass it onto our chat model to see how it performs.

In [31]:
# create a new user prompt
prompt = HumanMessage(
    content=augment_prompt(query)
)
# add to messages
messages.append(prompt)

res = chat.invoke(messages)

In [32]:
print(res.content)

A MatFormer (Matryoshka Transformer) is an elastic model architecture designed for flexible inference across different deployment constraints, incorporating nested Feed Forward Network layers within standard transformer blocks to enable real-time adjustment of parameter sizes. It allows the extraction of numerous smaller models that can operate efficiently on various devices without separate training, aiming at reducing latency and computational costs while optimizing for specific inference requirements.


We can continue with more questions. Let's try _without_ RAG first:

In [33]:
prompt = HumanMessage(
    content="What is SESNet?"
)

res = chat.invoke(messages + [prompt])
print(res.content)

MatFormer stands for Matryoshka Transformer, which introduces an elastic model architecture known as "Nested Feed Forward Network" layers within standard transformers to allow dynamic generation of numerous small sub-models during a single training run. This design helps address various deployment constraints by enabling real-time slicing and parameter resizing for different computational environments without separate trainings.


We haven't yet provided it with context, so it relies on the context it has. Let's try again but with RAG.

In [34]:
prompt = HumanMessage(
    content=augment_prompt(
        "What is SESNet?"
    )
)

res = chat.invoke(messages + [prompt])
print(res.content)

SESNet stands for Semantic-Efficient Self-Supervised Network, which was introduced as a method in Computer Vision tasks such as image classification and object detection that prioritizes capturing semantic concepts while reducing the need for extensive labeled data. The network uses self-supervised learning techniques to understand scene contexts without requiring detailed annotations of every aspect within an image or video frame.


Delete the index to save resources:

In [35]:
vector_store.delete_collection()